In [1]:
# === CELL 1: IMPORTS AND CONFIGURATION ===
import time
import random
import os
import re
import json
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

OUTPUT_CSV = "data/inc42_complaints.csv"
OUTPUT_JSON = "data/inc42.json"
TXT_DIR = "data/txt"
MAX_PAGES = 10
os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

def init_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=options)
    return driver

# ── ID Continuity ──────────────────────────────────────────────────────────
existing_max_id = 0
for csv_path in [
    "data/consumer_complaints.csv",
    "data/indiankanoon_complaints.csv",
    "data/medianama_complaints.csv",
    "data/reddit_complaints.csv",
    "data/entrackr_complaints.csv"  # may or may not exist
]:
    if os.path.exists(csv_path):
        df_check = pd.read_csv(csv_path)
        if "Unique ID" in df_check.columns:
            ids = df_check["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
            if not ids.empty:
                existing_max_id = max(existing_max_id, int(ids.max()))

# Also scan txt folder as safety net
for fname in os.listdir("data/txt"):
    match = re.search(r'NA-(\d+)\.txt', fname)
    if match:
        existing_max_id = max(existing_max_id, int(match.group(1)))

next_id = existing_max_id + 1
print(f"Starting Unique ID from: NA-{next_id:04d}")

Starting Unique ID from: NA-28130


In [2]:
# === CELL 2: KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [3]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["victim", "lost", "cheated", "defrauded",
                                       "money stolen", "account hacked", "fell for"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["beware", "warning", "alert", "avoid",
                                         "do not", "scam alert", "red flag"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def safe_save(df, csv_path, json_path, json_data):
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        print(f"  ✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️ Checkpoint save failed (data still in memory): {e}")

In [4]:
# === CELL 4: SELENIUM SEARCH FUNCTION ===
def search_inc42_selenium(driver, keyword, max_pages=10):
    results = []
    seen_urls = set()

    search_url = f"https://inc42.com/?s={keyword.replace(' ', '+')}"
    print(f"  Loading: {search_url}")
    driver.get(search_url)

    for page in range(max_pages):
        try:
            # Wait for Algolia to populate results
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "#inc-algolia-hits article, #inc-algolia-hits a[href*='inc42.com']"))
            )
            time.sleep(random.uniform(2.0, 4.0))  # extra wait for JS to finish

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            hits_div = soup.find('div', id='inc-algolia-hits')
            if not hits_div:
                break
            articles = hits_div.find_all('article')
            if not articles:
                print(f"  No articles rendered on page {page+1}")
                break
            for article in articles:
                a_tag = article.find('h2', class_='entry-title')
                a_tag = a_tag.find('a') if a_tag else article.find('a', href=re.compile(r'inc42\.com'))
                if not a_tag:
                    continue
                url = a_tag.get('href', '')
                title = a_tag.get_text(strip=True)
                date_span = article.find('span', class_='date')
                date = date_span.get_text(strip=True) if date_span else "Unknown Date"
                if url in seen_urls or len(title) < 5:
                    continue
                seen_urls.add(url)
                results.append({"title": title, "url": url, "date": date})

            print(f"  Page {page+1}: found {len(results)} results so far")

            # Try clicking next page button
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR,
                    "#inc-algolia-pagination .ais-Pagination-item--nextPage a, "
                    "#inc-algolia-pagination a[aria-label='Next'], "
                    "#inc-algolia-pagination .next a")
                if next_btn and next_btn.is_enabled():
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(random.uniform(3.0, 5.0))
                    # Wait for pagination to load
                    WebDriverWait(driver, 20).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "#inc-algolia-hits article"))
                    )
                else:
                    print(f"  No more pages after page {page+1}")
                    break
            except NoSuchElementException:
                print(f"  No next button found — end of results")
                break

        except TimeoutException:
            print(f"  Timeout on page {page+1} for '{keyword}'")
            break
        except Exception as e:
            print(f"  Error on page {page+1}: {e}")
            break

    return results

In [5]:
# === CELL 5: ARTICLE FULL TEXT FETCH FUNCTION ===
def fetch_inc42_article(driver, url):
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR,
                "div.td-post-content, div.entry-content, article"))
        )
        time.sleep(random.uniform(2.0, 4.0))

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Remove noise elements
        for tag in soup.find_all(['script', 'style', 'aside',
                                   'figure', 'nav', 'iframe']):
            tag.decompose()

        # Get article body — try multiple selectors
        content = (
            soup.find('div', class_='td-post-content') or
            soup.find('div', class_='entry-content') or
            soup.find('div', class_=re.compile(r'post.content|article.content|ais-hits--content', re.I)) or
            soup.find('article')
        )

        text = clean_text(content.get_text(separator='\n', strip=True)) if content else ""

        # Get author
        author_tag = (soup.find('a', rel='author') or
                      soup.find('div', class_='td-author-name') or
                      soup.find('span', class_=re.compile(r'author', re.I)))
        author = author_tag.get_text(strip=True) if author_tag else "Unknown"

        # Get date
        time_tag = soup.find('time')
        date = time_tag.get('datetime', '')[:10] if time_tag else "Unknown Date"

        return text, author, date

    except TimeoutException:
        print(f"  Timeout fetching article: {url}")
        return "", "Unknown", "Unknown Date"
    except Exception as e:
        print(f"  Error fetching {url}: {e}")
        return "", "Unknown", "Unknown Date"

In [7]:
# === CELL 6: MAIN SCRAPING LOOP ===
# Load existing data to avoid re-scraping
existing_urls = set()
existing_data = []
if os.path.exists(OUTPUT_JSON):
    try:
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            existing_data = json.load(f)
        for item in existing_data:
            if item.get("URL"):
                existing_urls.add(item["URL"])
    except json.JSONDecodeError:
        existing_data = []
print(f"Loaded {len(existing_urls)} existing URLs to skip")

today_date = datetime.now().strftime("%Y-%m-%d")
all_new_records = []
all_new_raw = []
new_count = 0

driver = init_driver()

try:
    for parent_category, subcategories in KEYWORD_TAXONOMY.items():
        for subcat, keywords in subcategories.items():
            for keyword in keywords:
                print(f"\n[*] '{keyword}' | {parent_category} > {subcat}")

                search_results = search_inc42_selenium(driver, keyword, max_pages=MAX_PAGES)

                for res in search_results:
                    url = res['url']
                    if url in existing_urls:
                        continue
                    existing_urls.add(url)

                    print(f"  Fetching: {url[:80]}")
                    full_text, author, date = fetch_inc42_article(driver, url)

                    # Use search result date if article date not found
                    if date == "Unknown Date" and res['date'] != "Unknown Date":
                        date = res['date']

                    title = res['title']
                    unique_id = f"NA-{next_id:04d}"
                    txt_filename = f"{unique_id}.txt"

                    # Save txt file immediately
                    txt_content = f"SOURCE: Inc42\n"
                    txt_content += f"TITLE: {title}\n"
                    txt_content += f"AUTHOR: {author}\n"
                    txt_content += f"DATE: {date}\n"
                    txt_content += f"URL: {url}\n\n"
                    txt_content += "--- ARTICLE TEXT ---\n\n"
                    txt_content += full_text if full_text else "[Article text unavailable]"

                    with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f:
                        f.write(txt_content)

                    record = {
                        "Unique ID": unique_id,
                        "Date of Collection": today_date,
                        "Collector Name": "Soubhik Sarkar",
                        "Source Platform": "Inc42",
                        "Source Publication": "inc42.com",
                        "Original Date": date,
                        "Title/Headline": title,
                        "URL": url,
                        "Search Query Used": keyword,
                        "Fraud Category": parent_category,
                        "Fraud Subcategory": subcat,
                        "Narrative Type": classify_narrative_type(full_text),
                        "TXT File Name": txt_filename,
                        "Notes": f"Author: {author}"
                    }

                    raw_item = {
                        "URL": url,
                        "Title/Headline": title,
                        "Original Date": date,
                        "Author": author,
                        "StructuredData": record
                    }

                    all_new_records.append(record)
                    all_new_raw.append(raw_item)
                    next_id += 1
                    new_count += 1

                    # Checkpoint save every 25 records
                    if new_count % 25 == 0:
                        df_temp = pd.DataFrame(all_new_records)
                        combined_raw = existing_data + all_new_raw
                        safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, combined_raw)

finally:
    driver.quit()
    print(f"\nDriver closed. Total new records: {new_count}")

Loaded 0 existing URLs to skip

[*] 'cyber crime' | General Cybercrime / Cyber Fraud Terms > General Cybercrime
  Loading: https://inc42.com/?s=cyber+crime
  Timeout on page 1 for 'cyber crime'

[*] 'cybercrime' | General Cybercrime / Cyber Fraud Terms > General Cybercrime
  Loading: https://inc42.com/?s=cybercrime
  Timeout on page 1 for 'cybercrime'

[*] 'cyber fraud' | General Cybercrime / Cyber Fraud Terms > General Cybercrime
  Loading: https://inc42.com/?s=cyber+fraud

Driver closed. Total new records: 0


KeyboardInterrupt: 

In [ ]:
# === CELL 7: FINAL SAVE ===
if all_new_records:
    df_new = pd.DataFrame(all_new_records)

    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.drop_duplicates(subset=["URL"], keep="last", inplace=True)
    else:
        df_combined = df_new

    temp_csv = OUTPUT_CSV + ".tmp"
    df_combined.to_csv(temp_csv, index=False, encoding="utf-8-sig")
    os.replace(temp_csv, OUTPUT_CSV)
    df_combined.to_excel(OUTPUT_CSV.replace(".csv", ".xlsx"), index=False)

    combined_raw = existing_data + all_new_raw
    json_dedup = {v["URL"]: v for v in combined_raw}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(list(json_dedup.values()), f, indent=4, ensure_ascii=False)

    print(f"✅ Saved {len(df_combined)} total records to {OUTPUT_CSV}")
    print(f"📁 TXT files in {TXT_DIR}")
    ids = df_combined["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
    print(f"IDs: NA-{int(ids.min()):04d} to NA-{int(ids.max()):04d}")
else:
    print("No new records to save.")

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

# Non-headless so we can see what's happening
options = webdriver.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")
# NO headless this time

driver = webdriver.Chrome(options=options)

try:
    driver.get("https://inc42.com/?s=UPI+fraud")
    
    # Just wait a flat 15 seconds and see what loads
    print("Waiting 15 seconds for page to load...")
    time.sleep(15)
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    hits = soup.find('div', id='inc-algolia-hits')
    
    print(f"Hits div found: {hits is not None}")
    if hits:
        text = hits.get_text(strip=True)
        print(f"Hits div text length: {len(text)}")
        print(f"First 500 chars: {text[:500]}")
        print(f"\nHTML preview:\n{hits.prettify()[:2000]}")
    
    # Also check what URL we actually ended up on
    print(f"\nFinal URL: {driver.current_url}")
    print(f"Page title: {driver.title}")

finally:
    driver.quit()

Waiting 15 seconds for page to load...
Hits div found: True
Hits div text length: 0
First 500 chars: 

HTML preview:
<div class="load-result" id="inc-algolia-hits">
</div>


Final URL: https://inc42.com/?s=UPI+fraud
Page title: You searched for UPI fraud - Inc42 Media


In [6]:
# === CELL 8: DIAGNOSTIC CELL (run this first before main loop) ===
# Test Selenium on Inc42 search
driver = init_driver()
try:
    driver.get("https://inc42.com/?s=UPI+fraud")
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "#inc-algolia-hits"))
    )
    time.sleep(5)  # wait for Algolia to load

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    hits = soup.find('div', id='inc-algolia-hits')

    if hits:
        print(f"✅ inc-algolia-hits div found!")
        print(f"Content preview:\n{hits.prettify()[:2000]}")
    else:
        print("❌ inc-algolia-hits div not found")
        print(soup.find('main').prettify()[:2000] if soup.find('main') else "No main found")
finally:
    driver.quit()

✅ inc-algolia-hits div found!
Content preview:
<div class="load-result" id="inc-algolia-hits">
</div>

